# Construção do MVP - Regressão Logística

In [1]:
import sys
from pathlib import Path

# Garante que a raiz do projeto está no sys.path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## O que Encontrar neste Notebook?

# Importando as Bibliotecas

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)

from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
)

from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# MLflow
# ATENCAO: o servidor do MLflow precisa estar rodando antes de executar este notebook.
# Em um terminal separado, execute:
#   mlflow server --host 127.0.0.1 --port 5000
# Sem isso, as celulas de tracking falharao com ConnectionRefusedError.
import hashlib
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Churn_Prediction_Baselines")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1775937613379, experiment_id='1', last_update_time=1775937613379, lifecycle_stage='active', name='Churn_Prediction_Baselines', tags={}, workspace='default'>

# Carregamentos dos Dados

In [3]:
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


# KPI's de Performance

## KPI Técnico - Recall (Sensibilidade)

O principal KPI técnico escolhido, dado a natureza do nosso problema de negócio, foi o **Recall.** Aqui o objetivo é simples, queremos identificar o seguinte cenário: **de todos os clientes que iam churnar, quantos o modelo conseguiu identificar corretamente.** 

Queremos reduzir ao máximo os casos de Falsos Negativos que representam clientes que churnaram (cancelaram) e não agimos, resultando na perda de receita.

Essa métrica é crítica para o negócio e permite responder perguntas como: ***"Quanto churn eu estou deixando passar?***"

Para a experimentação vamos considerar modelos com recall > 0.70.

## KPI's Técnicos Secundários

Outras métricas técnicas serão utilizadas para critério de possíveis desempates entre modelos e também para uma melhor interpretação do desempenho nosso modelo:

**`auc`** - avalia o quão bem o modelo consegue separar os clientes que vão churnar dos que não vão (buscamos algo acima de 0,70);

**`precision`** - avalia quantos clientes que foram marcados como churn de fato churnaram. Essa métrica tem um impacto direta no caso de falsos positivos e estabelece um trade-off com `recall`. Ela representa o custo desnecessário com retenção de clientes que não vão churnar;

**`f1-score`** - é uma métrica de equilíbrio entre `precision` e `recall`. Ela consegue balancear custo de retenção e perda de clientes, porém não considera o valor financeiro do cliente (assume custos fp e fn, aproximadamente iguais, o que quase nunca é verdade). Como já possuímos métricas financeiras, essa métrica será para uso secundário. Ela nos auxiliará na identificação do quão eficiente o modelo está conseguindo capturar churn.


## KPI de Negócio (definir)

As métricas técnicas presumem que todos os clientes são iguais do ponto de vista financeiro. E, sabemos que isso não ocorre dentro do nosso cenário real do negócio. Dessa forma, o uso dos KPIs econômicos nos permite priorizar clientes de maior valor, reduzir perda financeira real, balancear a relação custo-benefício e tomar decisões mais assertivas com base no negócio.

Em outras palavras, o modelo não só prevê churn, ele prioriza clientes com maior impacto financeiro e maximiza o retorno da retenção. Abaixo temos alguns KPIs econômicos que podemos utilizar para avaliar nosso modelo do ponto de vista do negócio:

- Receita Protegida (Somatória de Montlhy Charges dos TP);
- Receita Perdida (CLTV dos FN);
- Custo de Retenção (FP * Custo_unitário);
- Valor Liquído ou ROI - retorno real gerado pelo modelo (Receita Protegida - Custo de Retenção);
- CLTV Médio dos TP (qualidade dos clientes capturados);
- Taxa de Captura de Valor: quanto do valor em risco conseguimos capturar? (Receita Protegida / (Receita Protegida + Receita Perdida));
- ROI da Retenção ((receita protegida - custo)/custo)

# Pré-Processamento dos Dados

## Sanity Check

In [4]:
# Informações gerais sobre o dataset
print("=== INFORMAÇÕES GERAIS DO DATASET ===\n")
print(df.info())

# Análise de valores ausentes
print("=== ANÁLISE DE MISSING VALUES ===\n")

missing_values = pd.DataFrame({
    'Coluna': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

if len(missing_values) > 0:
    print(missing_values)

# Valores Duplicados
print("\n=== ANÁLISE DE VALORES DUPLICADOS ===\n")
duplicate_count = df.duplicated().sum()
print(f"Total de linhas duplicadas: {duplicate_count}")



=== INFORMAÇÕES GERAIS DO DATASET ===

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null 

## Dropando Variáveis

In [5]:
drop_cols = [
    'CustomerID',
    'Country',
    'State',
    'Lat Long',
    'Churn Label',
    'Churn Reason',
    'CLTV',
    'Latitude',
    'Longitude',
    'City',
    'Churn Score',
    'Zip Code',
    'Count'
]

df.drop(columns=drop_cols, inplace=True)

In [6]:
df.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,1
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,1


## Tratamento dos Dados

In [7]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [8]:
df['Total Charges'].isnull().sum()

np.int64(11)

In [9]:
df[df['Total Charges'].isna()][['Tenure Months', 'Monthly Charges']]

,Tenure Months,Monthly Charges
2234,0,52.55
2438,0,20.25
2568,0,80.85
2667,0,25.75
2856,0,56.05
4331,0,19.85
4687,0,25.35
5104,0,20.00
5719,0,19.70
6772,0,73.35


In [10]:
df['Total Charges'] = df['Total Charges'].fillna(0)

## Codificando as Variáveis

In [11]:
# Separando as Variáveis
target = 'Churn Value'
num_vars = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
cat_vars = [col for col in df.columns if df[col].dtype in ['str', 'object', 'category']]

print(f"Variáveis numéricas: {num_vars}")
print(f"Variáveis categóricas: {cat_vars}")

Variáveis numéricas: ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Value']
Variáveis categóricas: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


In [12]:
# dummificando as variáveis categóricas
dummy_vars = pd.get_dummies(df[cat_vars], drop_first=True, dtype=int)
dummy_vars.head()

,Gender_Male,Senior Citizen_Yes,Partner_Yes,Dependents_Yes,Phone Service_Yes,Multiple Lines_No phone service,Multiple Lines_Yes,Internet Service_Fiber optic,Internet Service_No,Online Security_No internet service,...,Streaming TV_No internet service,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Paperless Billing_Yes,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
1,0,0,0,1,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
2,0,0,0,1,1,0,1,1,0,0,...,0,1,0,1,0,0,1,0,1,0
3,0,0,1,1,1,0,1,1,0,0,...,0,1,0,1,0,0,1,0,1,0
4,1,0,0,1,1,0,1,1,0,0,...,0,1,0,1,0,0,1,0,0,0


In [13]:
# concatenando as variáveis numéricas e dummificadas
df_final = pd.concat([df[num_vars], dummy_vars], axis=1)
df_final.head()

,Tenure Months,Monthly Charges,Total Charges,Churn Value,Gender_Male,Senior Citizen_Yes,Partner_Yes,Dependents_Yes,Phone Service_Yes,Multiple Lines_No phone service,...,Streaming TV_No internet service,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Paperless Billing_Yes,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,2,53.85,108.15,1,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1
1,2,70.70,151.65,1,0,0,0,1,1,0,...,0,0,0,0,0,0,1,0,1,0
2,8,99.65,820.50,1,0,0,0,1,1,0,...,0,1,0,1,0,0,1,0,1,0
3,28,104.80,3046.05,1,0,0,1,1,1,0,...,0,1,0,1,0,0,1,0,1,0
4,49,103.70,5036.30,1,1,0,0,1,1,0,...,0,1,0,1,0,0,1,0,0,0


## Estratégia de Validação (StratifiedKFold)

In [14]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

## Separando as Bases

In [15]:
X = df_final.drop(columns=[target])
y = df_final[target]


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y  
)

# Baseline (DummyClassifier)

In [16]:
# Modelo baseline (Dummy)
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)


# Predições
y_train_pred = dummy.predict(X_train)
y_test_pred = dummy.predict(X_test)

# Para AUC (probabilidade)
y_train_proba = dummy.predict_proba(X_train)[:, 1]
y_test_proba = dummy.predict_proba(X_test)[:, 1]


# Métricas - Treino
metrics_train = {
    'recall': recall_score(y_train, y_train_pred),
    'precision': precision_score(y_train, y_train_pred, zero_division=0),
    'f1_score': f1_score(y_train, y_train_pred, zero_division=0),
    'auc': roc_auc_score(y_train, y_train_proba)
}


# Métricas - Teste
metrics_test = {
    'recall': recall_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred, zero_division=0),
    'f1_score': f1_score(y_test, y_test_pred, zero_division=0),
    'auc': roc_auc_score(y_test, y_test_proba)
}


# Overfitting (Recall)
recall_train = metrics_train['recall']
recall_test = metrics_test['recall']

# Evita divisão por zero
if recall_train == 0:
    overfitting_recall = 0.0
else:
    overfitting_recall = ((recall_train - recall_test) / recall_train) * 100


# Log estruturado
log_metrics = {
    'train': metrics_train,
    'test': metrics_test,
    'overfitting_recall_%': overfitting_recall
}


# Print formatado
print("===== BASELINE - DUMMY =====\n")

print(">> TREINO")
for k, v in metrics_train.items():
    print(f"{k}: {v:.4f}")

print("\n>> TESTE")
for k, v in metrics_test.items():
    print(f"{k}: {v:.4f}")

print("\n>> OVERFITTING (Recall)")
print(f"{overfitting_recall:.2f}%")


===== BASELINE - DUMMY =====

>> TREINO
recall: 0.0000
precision: 0.0000
f1_score: 0.0000
auc: 0.5000

>> TESTE
recall: 0.0000
precision: 0.0000
f1_score: 0.0000
auc: 0.5000

>> OVERFITTING (Recall)
0.00%


In [17]:
# MLflow — registrar DummyClassifier
dataset_hash = hashlib.md5(open("../data/raw/Telco_customer_churn.xlsx", "rb").read()).hexdigest()[:8]

with mlflow.start_run(run_name="DummyClassifier"):
    # Parâmetros
    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("test_size", 0.3)
    mlflow.log_param("dataset_version", dataset_hash)
    mlflow.log_param("n_samples_train", len(X_train))
    mlflow.log_param("n_features", X_train.shape[1])

    # Métricas — Treino
    mlflow.log_metric("train_recall",    metrics_train["recall"])
    mlflow.log_metric("train_precision", metrics_train["precision"])
    mlflow.log_metric("train_f1",        metrics_train["f1_score"])
    mlflow.log_metric("train_auc",       metrics_train["auc"])

    # Métricas — Teste
    mlflow.log_metric("test_recall",     metrics_test["recall"])
    mlflow.log_metric("test_precision",  metrics_test["precision"])
    mlflow.log_metric("test_f1",         metrics_test["f1_score"])
    mlflow.log_metric("test_auc",        metrics_test["auc"])
    mlflow.log_metric("overfitting_recall_pct", overfitting_recall)

    # Artefato
    mlflow.sklearn.log_model(dummy, "dummy_classifier")

print("DummyClassifier registrado no MLflow.")


2026/04/12 13:17:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 13:17:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run DummyClassifier at: http://localhost:5000/#/experiments/1/runs/5e486375b7ee404a914721b28303726d
🧪 View experiment at: http://localhost:5000/#/experiments/1
DummyClassifier registrado no MLflow.


# Pipeline da Regressão Logística

In [18]:
# Pipeline
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])



scoring = {
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)


# Agregação CV
cv_metrics = {
    'train': {
        'recall': np.mean(cv_results['train_recall']),
        'precision': np.mean(cv_results['train_precision']),
        'f1_score': np.mean(cv_results['train_f1']),
        'auc': np.mean(cv_results['train_roc_auc'])
    },
    'validation': {
        'recall': np.mean(cv_results['test_recall']),
        'precision': np.mean(cv_results['test_precision']),
        'f1_score': np.mean(cv_results['test_f1']),
        'auc': np.mean(cv_results['test_roc_auc'])
    }
}


# Fit final no treino
pipeline.fit(X_train, y_train)


# Predições
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

y_train_proba = pipeline.predict_proba(X_train)[:, 1]
y_test_proba = pipeline.predict_proba(X_test)[:, 1]


# Métricas - Treino
metrics_train = {
    'recall': recall_score(y_train, y_train_pred),
    'precision': precision_score(y_train, y_train_pred),
    'f1_score': f1_score(y_train, y_train_pred),
    'auc': roc_auc_score(y_train, y_train_proba)
}


# Métricas - Teste
metrics_test = {
    'recall': recall_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred),
    'f1_score': f1_score(y_test, y_test_pred),
    'auc': roc_auc_score(y_test, y_test_proba)
}


# Overfitting (Recall)
recall_train = metrics_train['recall']
recall_test = metrics_test['recall']

if recall_train == 0:
    overfitting_recall = 0.0
else:
    overfitting_recall = ((recall_train - recall_test) / recall_train) * 100


# Log estruturado
log_metrics = {
    'cv': cv_metrics,
    'train': metrics_train,
    'test': metrics_test,
    'overfitting_recall_%': overfitting_recall
}


# Print formatado
print("===== LOGISTIC REGRESSION (PIPELINE + CV) =====\n")

print(">> CROSS VALIDATION (MÉDIA)")
for split in ['train', 'validation']:
    print(f"\n[{split.upper()}]")
    for k, v in cv_metrics[split].items():
        print(f"{k}: {v:.4f}")

print("\n>> TREINO (FULL FIT)")
for k, v in metrics_train.items():
    print(f"{k}: {v:.4f}")

print("\n>> TESTE")
for k, v in metrics_test.items():
    print(f"{k}: {v:.4f}")

print("\n>> OVERFITTING (Recall)")
print(f"{overfitting_recall:.2f}%")


===== LOGISTIC REGRESSION (PIPELINE + CV) =====

>> CROSS VALIDATION (MÉDIA)

[TRAIN]
recall: 0.8185
precision: 0.5328
f1_score: 0.6454
auc: 0.8624

[VALIDATION]
recall: 0.8134
precision: 0.5308
f1_score: 0.6422
auc: 0.8573

>> TREINO (FULL FIT)
recall: 0.8211
precision: 0.5325
f1_score: 0.6460
auc: 0.8623

>> TESTE
recall: 0.7950
precision: 0.5156
f1_score: 0.6255
auc: 0.8531

>> OVERFITTING (Recall)
3.18%


In [19]:
# MLflow — registrar Regressão Logística
lr_params = pipeline.named_steps["model"].get_params()

with mlflow.start_run(run_name="LogisticRegression"):
    # Parâmetros
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("C", lr_params["C"])
    mlflow.log_param("max_iter", lr_params["max_iter"])
    mlflow.log_param("class_weight", lr_params["class_weight"])
    mlflow.log_param("random_state", lr_params["random_state"])
    mlflow.log_param("cv_folds", 10)
    mlflow.log_param("test_size", 0.3)
    mlflow.log_param("dataset_version", dataset_hash)
    mlflow.log_param("n_samples_train", len(X_train))
    mlflow.log_param("n_features", X_train.shape[1])

    # Métricas — Cross Validation
    mlflow.log_metric("cv_recall",    cv_metrics["validation"]["recall"])
    mlflow.log_metric("cv_precision", cv_metrics["validation"]["precision"])
    mlflow.log_metric("cv_f1",        cv_metrics["validation"]["f1_score"])
    mlflow.log_metric("cv_auc",       cv_metrics["validation"]["auc"])

    # Métricas — Treino
    mlflow.log_metric("train_recall",    metrics_train["recall"])
    mlflow.log_metric("train_precision", metrics_train["precision"])
    mlflow.log_metric("train_f1",        metrics_train["f1_score"])
    mlflow.log_metric("train_auc",       metrics_train["auc"])

    # Métricas — Teste
    mlflow.log_metric("test_recall",     metrics_test["recall"])
    mlflow.log_metric("test_precision",  metrics_test["precision"])
    mlflow.log_metric("test_f1",         metrics_test["f1_score"])
    mlflow.log_metric("test_auc",        metrics_test["auc"])
    mlflow.log_metric("overfitting_recall_pct", overfitting_recall)

    # Artefato
    mlflow.sklearn.log_model(pipeline, "logistic_regression_pipeline")

print("Regressão Logística registrada no MLflow.")


2026/04/12 13:17:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 13:17:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogisticRegression at: http://localhost:5000/#/experiments/1/runs/43a8cef3ca59476c8fc7812cc5eb12d1
🧪 View experiment at: http://localhost:5000/#/experiments/1
Regressão Logística registrada no MLflow.


# Pipeline MLP - PyTorch

## Configurações da Rede Neural

In [20]:
# Configuração
EPOCHS = 30
BATCH_SIZE = 32
LR = 0.001  # taxa de aprendizado
WD = 1e-5   # weight decay (regularização)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
THRESHOLD = 0.5   


## Arquitetura do MLP

In [21]:
# Arquitetura MLP extraída para src/models/mlp.py
from src.models.mlp import MLP, evaluate
from src.models.mlp import train_epoch as train_model


## Funções Auxiliares

In [22]:
# Métricas extraídas para src/evaluation/metrics.py
from src.evaluation.metrics import avg_metrics, compute_metrics


## Fluxo de Treinamento e Validação

## Validação Cruzada

In [23]:
# Cross-validation
cv_train_metrics = []
cv_val_metrics = []

X_train = np.asarray(X_train)
y_train = np.asarray(y_train)

for train_idx, val_idx in cv.split(X_train, y_train):
    
    # Split dos dados
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    # Escalonamento dos Dados
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)

    # calculo dos pesos para a classe positiva (churn) e negativa (não churn)
    pos = y_tr.sum()
    neg = len(y_tr) - pos

    pos_weight = torch.tensor(
        [neg / pos if pos > 0 else 1.0],
        dtype=torch.float32
    ).to(DEVICE)
    
    # tensor
    train_dataset = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32)
    )

    val_dataset   = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32)
    )

    # dataloader
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    
    # Modelo
    model = MLP(input_dim=X_train.shape[1]).to(DEVICE)
    
    # Otimizador e critério
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


    #Treino
    for epoch in range(10):
        train_loss = train_model(model, train_loader, optimizer, criterion)
        val_loss, _ = evaluate(model, val_loader, criterion)

        print(f"Epoch {epoch+1} | Train_loss: {train_loss:.4f} | Val_loss: {val_loss:.4f}")

    # métricas finais do fold
    _, train_metrics = evaluate(model, train_loader, criterion)
    _, val_metrics = evaluate(model, val_loader, criterion)

    cv_train_metrics.append(train_metrics)
    cv_val_metrics.append(val_metrics)



# Agregação CV
cv_train_mean = avg_metrics(cv_train_metrics)
cv_val_mean = avg_metrics(cv_val_metrics)


cv_results_df = pd.DataFrame({
    "Train": cv_train_mean,
    "Validation": cv_val_mean
})

cv_results_df["Gap (%)"] = (
    (cv_results_df["Train"] - cv_results_df["Validation"]) 
    / cv_results_df["Train"]
) * 100

print("\n===== CV RESULTS =====")
display(cv_results_df.round(4))

Epoch 1 | Train_loss: 0.8069 | Val_loss: 0.7086
Epoch 2 | Train_loss: 0.6971 | Val_loss: 0.6965
Epoch 3 | Train_loss: 0.6792 | Val_loss: 0.7020
Epoch 4 | Train_loss: 0.6693 | Val_loss: 0.7005
Epoch 5 | Train_loss: 0.6639 | Val_loss: 0.7027
Epoch 6 | Train_loss: 0.6589 | Val_loss: 0.7018
Epoch 7 | Train_loss: 0.6555 | Val_loss: 0.6964
Epoch 8 | Train_loss: 0.6505 | Val_loss: 0.7018
Epoch 9 | Train_loss: 0.6474 | Val_loss: 0.7036
Epoch 10 | Train_loss: 0.6453 | Val_loss: 0.6993
Epoch 1 | Train_loss: 0.8048 | Val_loss: 0.7636
Epoch 2 | Train_loss: 0.6965 | Val_loss: 0.7573
Epoch 3 | Train_loss: 0.6786 | Val_loss: 0.7351
Epoch 4 | Train_loss: 0.6681 | Val_loss: 0.7393
Epoch 5 | Train_loss: 0.6634 | Val_loss: 0.7365
Epoch 6 | Train_loss: 0.6549 | Val_loss: 0.7265
Epoch 7 | Train_loss: 0.6496 | Val_loss: 0.7384
Epoch 8 | Train_loss: 0.6469 | Val_loss: 0.7333
Epoch 9 | Train_loss: 0.6428 | Val_loss: 0.7247
Epoch 10 | Train_loss: 0.6401 | Val_loss: 0.7427
Epoch 1 | Train_loss: 0.7852 | Val_los

,Train,Validation,Gap (%)
recall,0.8282,0.7981,3.6298
precision,0.5496,0.5304,3.4917
f1,0.6605,0.6366,3.6159
auc,0.8771,0.8570,2.2891


## MLflow — Registrar MLP

In [24]:
# MLflow — registrar MLP
import mlflow.pytorch

# Treino final no conjunto completo de treino
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train)
X_test_scaled  = scaler_final.transform(np.asarray(X_test))

pos = y_train.sum()
neg = len(y_train) - pos
pos_weight_final = torch.tensor(
    [neg / pos if pos > 0 else 1.0], dtype=torch.float32
).to(DEVICE)

train_dataset_final = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(np.asarray(y_train), dtype=torch.float32)
)
test_dataset_final = TensorDataset(
    torch.tensor(X_test_scaled, dtype=torch.float32),
    torch.tensor(np.asarray(y_test), dtype=torch.float32)
)
train_loader_final = DataLoader(train_dataset_final, batch_size=BATCH_SIZE, shuffle=True)
test_loader_final  = DataLoader(test_dataset_final,  batch_size=BATCH_SIZE)

mlp_final = MLP(input_dim=X_train_scaled.shape[1]).to(DEVICE)
optimizer_final = optim.Adam(mlp_final.parameters(), lr=LR, weight_decay=WD)
criterion_final = nn.BCEWithLogitsLoss(pos_weight=pos_weight_final)

for epoch in range(EPOCHS):
    train_model(mlp_final, train_loader_final, optimizer_final, criterion_final)

_, metrics_train_mlp = evaluate(mlp_final, train_loader_final, criterion_final, threshold=THRESHOLD)
_, metrics_test_mlp  = evaluate(mlp_final, test_loader_final,  criterion_final, threshold=THRESHOLD)

overfitting_recall_mlp = (
    (metrics_train_mlp["recall"] - metrics_test_mlp["recall"])
    / metrics_train_mlp["recall"] * 100
    if metrics_train_mlp["recall"] > 0 else 0.0
)

with mlflow.start_run(run_name="MLP"):
    # Params
    mlflow.log_param("model",           "MLP")
    mlflow.log_param("hidden_dim",      64)
    mlflow.log_param("epochs",          EPOCHS)
    mlflow.log_param("batch_size",      BATCH_SIZE)
    mlflow.log_param("lr",              LR)
    mlflow.log_param("weight_decay",    WD)
    mlflow.log_param("threshold",       THRESHOLD)
    mlflow.log_param("cv_folds",        10)
    mlflow.log_param("test_size",       0.3)
    mlflow.log_param("dataset_version", dataset_hash)
    mlflow.log_param("n_samples_train", len(X_train))
    mlflow.log_param("n_features",      X_train_scaled.shape[1])

    # Metricas CV
    mlflow.log_metric("cv_recall",    cv_val_mean["recall"])
    mlflow.log_metric("cv_precision", cv_val_mean["precision"])
    mlflow.log_metric("cv_f1",        cv_val_mean["f1"])
    mlflow.log_metric("cv_auc",       cv_val_mean["auc"])

    # Metricas Treino
    mlflow.log_metric("train_recall",    metrics_train_mlp["recall"])
    mlflow.log_metric("train_precision", metrics_train_mlp["precision"])
    mlflow.log_metric("train_f1",        metrics_train_mlp["f1"])
    mlflow.log_metric("train_auc",       metrics_train_mlp["auc"])

    # Metricas Teste
    mlflow.log_metric("test_recall",    metrics_test_mlp["recall"])
    mlflow.log_metric("test_precision", metrics_test_mlp["precision"])
    mlflow.log_metric("test_f1",        metrics_test_mlp["f1"])
    mlflow.log_metric("test_auc",       metrics_test_mlp["auc"])
    mlflow.log_metric("overfitting_recall_pct", overfitting_recall_mlp)

    mlflow.pytorch.log_model(mlp_final, "mlp_baseline")

print("MLP registrada no MLflow.")

2026/04/12 13:18:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 13:18:02 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


🏃 View run MLP at: http://localhost:5000/#/experiments/1/runs/7949ae44e2c9403ea02c5833e0fce9f6
🧪 View experiment at: http://localhost:5000/#/experiments/1
MLP registrada no MLflow.


## Conclusão

# Próximos Passos

- Prosseguir para Etapa de Experimentação do Modelo de Regressão, realizando feature engineering;